In [ ]:
# =================================================================
# PROYECTO INTEGRADOR: MACHINE LEARNING - FASE 2A
# GRUPO 2: Analisis de Satisfaccion en Productos de Oficina
# Notebook: Modelado Clasico (LogReg, Random Forest, LightGBM, XGBoost)
# =================================================================

import sys
from pathlib import Path
import pandas as pd
import numpy as np
import gc
import pickle
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, precision_recall_fscore_support

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/ML/proyecto_integrador')
else:
    BASE = Path('..')

DATA_DIR = BASE / 'data'
REPORTS_DIR = BASE / 'reports'
MODELS_DIR = BASE / 'models'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, precision_recall_fscore_support
import lightgbm as lgb
import xgboost as xgb


In [ ]:
# --- BLOQUE 2: CARGA DE DATOS ---
# Cargamos el dataset canonico generado por f1_eda_definitivo (2.5M filas balanceadas 500k/rating)
# Remapeamos rating 1-5 a 3 clases: Negativo (1-2), Neutro (3), Positivo (4-5)
print('Cargando dataset canonico desde f1_eda_definitivo...')

CANONICAL_PATH = DATA_DIR / 'office_products_balanced.parquet'

try:
    df = pd.read_parquet(CANONICAL_PATH, columns=['text', 'rating', 'sentiment'])
except FileNotFoundError:
    print(f'[ERROR] Dataset canonico no encontrado en {CANONICAL_PATH}')
    print('Ejecute primero f1_eda_definitivo.ipynb en Colab para generarlo.')
    raise
except Exception as e:
    print(f'[ERROR] No se pudo cargar el dataset: {e}')
    raise

# Si no existe columna sentiment (del EDA), remapeamos desde rating
if 'sentiment' not in df.columns:
    df['sentiment'] = df['rating'].map(lambda r: 0 if r <= 2 else (1 if r == 3 else 2))

# word_count filter: purgar reseñas telegraficas
df['word_count'] = df['text'].astype(str).str.split().str.len()
df = df[df['word_count'] >= 5]
print(f'Registros tras filtro word_count >= 5: {len(df):,}')

# Mostrar distribucion de clases
print('\nDistribucion de sentimiento:')
print(df['sentiment'].value_counts().sort_index())


In [ ]:
# --- BLOQUE 3: DIVISIÓN DE DATOS (PREVENCIÓN DE LEAKAGE) ---
# Justificación (Data Preparation): Dividimos ANTES de vectorizar. 
# Si vectorizamos antes de dividir, el vocabulario del set de prueba 
# "contaminaría" el entrenamiento (Data Leakage).
print(" Dividiendo datos en Train y Test (80/20)...")
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['text_cleaned'], df['sentiment_class'], 
    test_size=0.20, random_state=42, stratify=df['sentiment_class']
)

# Liberar RAM
del df
gc.collect()

In [ ]:
# --- BLOQUE 4: VECTORIZACIÓN TF-IDF ---
# Justificación (Data Preparation): Usamos min_df=5 para ignorar palabras
# que aparecen menos de 5 veces (errores ortográficos), reduciendo ruido.
print(" Entrenando TF-IDF Vectorizer...")
vectorizer = TfidfVectorizer(
    max_features=15000, 
    ngram_range=(1, 2), 
    stop_words='english',
    min_df=5 
)

# Fit SOLO en train, transform en test
X_train = vectorizer.fit_transform(X_train_text.astype(str))
X_test = vectorizer.transform(X_test_text.astype(str))

print(f" Matriz de entrenamiento: {X_train.shape}")

In [ ]:
print('\nEntrenando Modelo 1: Regresion Logistica (Baseline)...')
log_model = LogisticRegression(max_iter=1000, C=0.5, class_weight='balanced', random_state=42, n_jobs=-1)
log_model.fit(X_train, y_train)
y_pred_log = log_model.predict(X_test)


In [ ]:
print('Entrenando Modelo 2: Random Forest...')
rf_model = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

print('Entrenando Modelo 3: LightGBM...')
lgb_model = lgb.LGBMClassifier(class_weight='balanced', random_state=42, verbose=-1)
lgb_model.fit(X_train, y_train)
y_pred_lgb = lgb_model.predict(X_test)


In [ ]:
gc.collect()

print('Entrenando Modelo 4: XGBoost...')

# Compute class weights for XGBoost
from sklearn.utils.class_weight import compute_class_weight
classes = np.unique(y_train)
cw = compute_class_weight('balanced', classes=classes, y=y_train)
class_weights_dict = {int(c): float(w) for c, w in zip(classes, cw)}

xgb_model = xgb.XGBClassifier(
    n_estimators=600,
    max_depth=5,
    learning_rate=0.05,
    objective='multi:softprob',
    num_class=3,
    random_state=42,
    eval_metric='mlogloss'
)

# Sample weights para balance de clases
sample_weights = np.array([class_weights_dict[y] for y in y_train])
xgb_model.fit(X_train, y_train, sample_weight=sample_weights)
y_pred_xgb = xgb_model.predict(X_test)

print('Entrenamiento completado.')


In [ ]:
# --- BLOQUE 6: BENCHMARK CON AUTOML (LAZYPREDICT) ---
# Justificacion: LazyPredict ejecuta 27+ clasificadores sobre la misma particion
# y reporta F1-score, Accuracy, Precision, Recall para comparacion objetiva.
print('\nIniciando Benchmark AutoML (LazyPredict)...')

from lazypredict.Supervised import LazyClassifier

clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)
models_df, predictions = clf.fit(X_train, X_test, y_train, y_test)

print('\nTop 10 modelos segun LazyPredict:')
print(models_df.head(10).to_string())

# Guardar resultados LazyPredict
lazy_path = REPORTS_DIR / 'lazypredict_results.csv'
models_df.to_csv(lazy_path)
print(f'\nResultados completos guardados en {lazy_path}')


In [ ]:
# --- BLOQUE 7: EVALUACION Y COMPARACION (CRISP-DM: Evaluation) ---
# Justificacion: Misma particion train/test. Métrica rectora: F1-macro.
print('\n' + '='*60)
print('EVALUACION FINAL - F1-macro como metrica rectora')
print('='*60)

models = {
    'Logistic Regression (Baseline)': y_pred_log,
    'Random Forest': y_pred_rf,
    'LightGBM': y_pred_lgb,
    'XGBoost': y_pred_xgb
}

results = []
class_labels = ['Negativo', 'Neutro', 'Positivo']

for name, preds in models.items():
    f1_macro = f1_score(y_test, preds, average='macro')
    acc = accuracy_score(y_test, preds)
    precision_macro, recall_macro, _, _ = precision_recall_fscore_support(y_test, preds, average='macro')
    cm = confusion_matrix(y_test, preds).tolist()

    p_class, r_class, f1_class, _ = precision_recall_fscore_support(y_test, preds, labels=[0, 1, 2])

    print(f"\n{'='*50}")
    print(f' {name}')
    print(f"{'='*50}")
    print(f'  F1-macro:  {f1_macro:.4f}')
    print(f'  Accuracy:  {acc:.4f}')
    print(f'  Precision macro: {precision_macro:.4f}')
    print(f'  Recall macro:    {recall_macro:.4f}')
    print('\n  Classification Report:')
    print(classification_report(y_test, preds, target_names=class_labels, digits=4))

    results.append({
        'model_name': name,
        'model_type': name.lower().replace(' ', '_').replace('(', '').replace(')', ''),
        'f1_macro': round(f1_macro, 4),
        'precision_macro': round(precision_macro, 4),
        'recall_macro': round(recall_macro, 4),
        'accuracy': round(acc, 4),
        'confusion_matrix': cm,
        'per_class': {
            cl: {
                'precision': round(p_class[i], 4),
                'recall': round(r_class[i], 4),
                'f1': round(f1_class[i], 4)
            } for i, cl in enumerate(class_labels)
        }
    })

# Exportar metricas a JSON
report_path = REPORTS_DIR / 'metrics_fase2.json'
try:
    with open(report_path, 'w') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f'\n[METRICAS] Exportadas a {report_path}')
except Exception as e:
    print(f'[ERROR] No se pudo exportar metricas: {e}')

print('\n' + '='*60)
print('ANALISIS DE ERROR - Por clase')
print('='*60)
for name, preds in models.items():
    print(f'\n--- {name} ---')
    cm = confusion_matrix(y_test, preds)
    for i, cl in enumerate(class_labels):
        total_real = cm[i].sum()
        correctos = cm[i][i]
        print(f'  {cl:12s}: {correctos:5d}/{total_real} correctos ({correctos/total_real*100:.1f}%)')
        errores = [(j, cm[i][j]) for j in range(3) if j != i]
        for j, cnt in sorted(errores, key=lambda x: -x[1]):
            print(f'           -> {cnt:5d} clasificados como {class_labels[j]}')
